<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefoerdert vom Bundesministerium fuer Forschung, Technologie und Raumfahrt.">
</div>
<h1> Workshop: Rapid Prototyping via Vibe Coding
<h2> Part 1 &mdash; context-mode
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
    <h2> Part 1 &mdash; context-mode</h2>
</div>

**context-mode** sandboxes the noisy half of an agentic session &mdash; tool outputs &mdash; so they don't flood Claude's context window.

Repository: <https://github.com/mksglu/context-mode> &middot; License: Elastic 2.0 (source-available).

The pitch, in one screenshot's worth of text: a single Playwright page snapshot costs ~56 KB. If Claude takes three snapshots while debugging a UI, that's 168 KB of markup in the window &mdash; competing with *your prompts and its own reasoning*. context-mode captures the snapshot, indexes it, and lets Claude `ctx_search` the index instead of reading the whole thing.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. What problem it solves</h3>
</div>

Three failure modes context-mode addresses:

1. **Tool output floods context.** Playwright snapshots, `kubectl get all -A`, large log files, web-scrape HTML &mdash; each reads cheap but lands heavy.
2. **Auto-compaction loses working state.** When the context window fills, Claude summarises. Summaries drop specifics. Your debugging session forgets the exact error message.
3. **Tokens spent on data, not code.** Large tool outputs push the model toward *reading* rather than *generating*. Performance degrades with context fullness; outputs get worse at the exact moment you need good code.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. What it does</h3>
</div>

- **Sandboxed execution** &mdash; run a command, capture its stdout, index it in a local SQLite store. Only a short receipt enters the context.
- **Indexed search** &mdash; Claude uses `ctx_search "..."` to pull *just* the relevant slice.
- **URL fetching** &mdash; `ctx_fetch_and_index URL` stores large documentation pages locally and exposes them through the same search.
- **Stats and diagnostics** &mdash; `/ctx-stats`, `/ctx-doctor` slash commands to see how much context you've saved and whether the plugin is wired up correctly.

The project claims up to **98% reduction** in context consumed by tool output. Your mileage varies by task; expect dramatic savings on any browser-automation or log-analysis workflow.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Install</h3>
</div>

Recommended: install as a Claude Code plugin.

```bash
/plugin marketplace add mksglu/context-mode
/plugin install context-mode@context-mode
```

Both commands work inside a running Claude Code session or as `claude plugin ...` from the shell.

Alternative for non-Claude-Code agents (Gemini CLI, Cursor, VS Code Copilot, Codex CLI, Zed, etc.): MCP-only mode:

```bash
claude mcp add context-mode -- npx -y context-mode
```

#### Prerequisites

- **Node.js &ge; 18** (22+ recommended on Linux).
- **better-sqlite3** (auto-built on install; precompiled on most systems).
- *(optional)* [**Bun**](https://bun.sh) &mdash; autodetected; gives 3&ndash;5x faster JS/TS execution for sandboxed runs.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Usage</h3>
</div>

Once installed, Claude will reach for context-mode's tools automatically when it detects a noisy command. You can also invoke them directly:

```bash
# Run a command in the sandbox; only stdout enters context.
ctx_execute "node script.js"

# Search what has been indexed.
ctx_search "authentication pattern"

# Fetch a URL and index it; don't dump the HTML into context.
ctx_fetch_and_index "https://docs.example.com/v2/webhooks"

# See how much context you're saving.
ctx stats
```

Slash commands inside a Claude Code session:

```text
/ctx-stats   # summary of savings across the current session
/ctx-doctor  # health-check: marketplace added, plugin installed, hooks wired
```

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. When it's worth it</h3>
</div>

- **Browser automation** (Playwright / Puppeteer). The snapshot sizes alone justify install.
- **Log / metrics analysis.** `kubectl logs`, `journalctl`, `tail -n 10000 ...`.
- **Web scraping / documentation lookup.** Index large docs once, query repeatedly.
- **Large file inspection** when you need *some* of the content but not all of it.

Skip it for:

- Pure refactoring inside a small codebase where you already know the files.
- One-off small-output commands (`git status`, `ls`).
- Work that depends on Claude seeing the raw text of a tool output verbatim (e.g. diff review).

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. License note</h3>
</div>

**Elastic License 2.0** is *source-available*, not OSI-open-source. It permits use, modification, and distribution, but prohibits hosted-SaaS redistribution and removing the licensing notices. Fine for internal and client work; if you're planning a product whose core is context-mode, check the license first.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>7. How this composes with the rest of Part 1</h3>
</div>

- **With [`docling-toolkit`](w1_03_docling_toolkit.ipynb):** Docling emits large intermediate JSONL. Let context-mode sandbox the extraction step so only the summary receipt lands in Claude's context.
- **With [`caveman`](w1_06_caveman.ipynb):** context-mode shrinks the *input* side of the transcript (tool output); caveman shrinks the *output* side (Claude's prose). Together they stretch the window on both axes.
- **With [`gepa`](w1_04_gepa.ipynb):** when optimising an agent whose traces run through context-mode, feed `ctx stats` into the GEPA metric so the optimiser can reward context-efficient candidates.